# Elastic Net sensitivity analysis

This notebook is a parallel of `prediction_model.ipynb` with **ElasticNetCV** substituted for `RidgeCV`. It serves as the sensitivity analysis reported in the supplement: it verifies that the primary findings are not specific to the L2 penalty in ridge regression by re-running the identical Y1 → Y2 pipeline with an L1+L2 mixed penalty.

**ElasticNetCV configuration.** `l1_ratio ∈ {0.10, 0.30, 0.50, 0.70, 0.90, 0.95, 0.99}` with 100 data-driven alphas per `l1_ratio` (sklearn's default path computation: `α_max` is the smallest alpha that drives all coefficients to zero, and the grid spans `[α_max·1e-3, α_max]` on a log scale). The endpoints `l1_ratio = 0` (pure ridge — already the primary model) and `l1_ratio = 1` (pure lasso) are excluded.

**Numerical caveat.** For some outcomes the inner-CV-selected ElasticNet driving-only model may shrink all 24 driving coefficients to zero, in which case its predictions reduce to the Y1 training mean and become identical to the constant-mean baseline. In that situation the paired t-test on `sq_err_driving` vs. `sq_err_baseline` is undefined (zero variance of paired differences); the permutation test correctly returns p = 1. 

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from statsmodels.stats.multitest import multipletests
from tqdm.notebook import tqdm

In [2]:
ivs = ['DaysDriving','Miles_n','Trips','TripMinutes_n','TripChains',
       'MilesPerTrip_n','MinutesPerTrip_n','MilesPerChain_n','MinutesPerChain_n',
       'Average_speed','LeftTurnCount','RightToLeftTurnRatio_n',
       'TripsAMPeak','PercentTripsAMPeak_n','TripsPMPeak','PercentTripsPMPeak_n',
       'TripsAtNight','PercentTripsAtNight_n','TripsLt15Miles','PercentDistLt15Miles_n',
       'TripsVgt60','PercentTripsVgt60_n','SpeedingPer1000Miles','DecelerationPer1000Miles']

dvs = ['LIFESATISFACTION','cog','phys','fati','socialrole','iso','info','emo','ins','dep','anx','ang']
demos_cat = ['Site','GENDER','RACE_ETH','WORK','MARRIAGE']
demos_num = ['Age','EDUCATION','INCOME']  # ordinal-coded but treated as numeric for consistency with R analyses

In [3]:
y1 = pd.read_csv('../data/y1_subject_level.csv')
y2 = pd.read_csv('../data/y2_subject_level.csv')

for df in (y1, y2):
    for col in ['XID'] + demos_cat:
        if col in df.columns:
            df[col] = df[col].astype('category')
# Outcomes are NOT scaled: MSE is reported on the original PROMIS / life-satisfaction scale.

print('Y1 file rows =', len(y1))
print('Y2 file rows =', len(y2))
print('Subjects in both Y1 and Y2:',
      len(set(y1['XID'].astype(str)).intersection(set(y2['XID'].astype(str)))))

Y1 file rows = 2990
Y2 file rows = 2990
Subjects in both Y1 and Y2: 2990


In [4]:
def build_preprocessor(numeric_cols, categorical_cols):
    """Per-column-type preprocessor with imputation. Identical to the primary
    `prediction_model.ipynb`.
    Numeric: mean-impute -> standard-scale.
    Categorical: mode-impute -> one-hot (drop first; handle_unknown='ignore').
    All imputers and the scaler are fit on training data only; .transform() on
    Y2 therefore uses Y1-derived statistics to fill in any missing Y2 predictors."""
    numeric_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        ('scale',  StandardScaler())])
    categorical_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe',    OneHotEncoder(drop='first', handle_unknown='ignore'))])
    transformers = []
    if numeric_cols:
        transformers.append(('num', numeric_pipe, numeric_cols))
    if categorical_cols:
        transformers.append(('cat', categorical_pipe, categorical_cols))
    return ColumnTransformer(transformers)

In [5]:
def train_y1_predict_y2(y1_df, y2_df, dv, id_col,
                        numeric_ivs, categorical_ivs,
                        l1_ratios=None, n_alphas=100,
                        random_state=0, inner_splits=10):
    """Fit ElasticNet on Y1 with inner 10-fold CV for (alpha, l1_ratio);
    predict on Y2. Parallel to `train_y1_predict_y2` in the primary notebook,
    with `RidgeCV(alphas=...)` replaced by `ElasticNetCV(l1_ratio=..., alphas=...)`.

    Preprocessing notes:
      * The imputers and the scaler inside the Pipeline are fit on Y1 only.
      * At predict time, Pipeline.predict() calls .transform() on Y2, which
        applies the Y1-fit statistics. Any Y2 subject missing a numeric
        predictor receives the Y1 column mean for that predictor; any Y2
        subject missing a categorical predictor receives the Y1 mode.
      * Subjects with missing y are dropped from train/test as appropriate
        (no squared error is defined without a true value).

    Returns one row per Y2 subject with non-missing y: id, y_true, y_pred,
    squared error, and the chosen alpha and l1_ratio (constant within a
    single Y1 fit)."""
    if l1_ratios is None:
        l1_ratios = [0.10, 0.30, 0.50, 0.70, 0.90, 0.95, 0.99]

    y1_use = y1_df.dropna(subset=[dv]).copy()
    y2_use = y2_df.dropna(subset=[dv]).copy()

    cols = list(numeric_ivs) + list(categorical_ivs)
    X_train = y1_use[cols]; y_train = y1_use[dv].values
    X_test  = y2_use[cols]; y_test  = y2_use[dv].values
    ids_test = y2_use[id_col].values

    preprocessor = build_preprocessor(list(numeric_ivs), list(categorical_ivs))
    inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)
    model = Pipeline([
        ('pre', preprocessor),
        ('reg', ElasticNetCV(l1_ratio=l1_ratios, alphas=n_alphas,
                             cv=inner_cv, max_iter=20000, n_jobs=1))])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    alpha_chosen    = float(model.named_steps['reg'].alpha_)
    l1_ratio_chosen = float(model.named_steps['reg'].l1_ratio_)

    return pd.DataFrame({
        'subject':  ids_test,
        'y_true':   y_test,
        'y_pred':   y_pred,
        'sq_err':   (y_test - y_pred) ** 2,
        'alpha':    alpha_chosen,
        'l1_ratio': l1_ratio_chosen,
    })


def baseline_y1_predict_y2(y1_df, y2_df, dv, id_col):
    """Constant-prediction baseline: predict every Y2 subject as the Y1
    training mean of the outcome. Identical to the primary notebook."""
    y1_use = y1_df.dropna(subset=[dv])
    y2_use = y2_df.dropna(subset=[dv]).copy()
    mu = y1_use[dv].mean()
    y_test = y2_use[dv].values
    return pd.DataFrame({
        'subject': y2_use[id_col].values,
        'y_true':  y_test,
        'y_pred':  mu,
        'sq_err':  (y_test - mu) ** 2,
    })


def paired_perm_test_mean_diff(diff, n_perm=10000, alternative='less',
                               random_state=0):
    """Permutation test on the mean of paired differences via sign-flipping.
    Identical to the primary notebook."""
    rng = np.random.default_rng(random_state)
    d = np.asarray(diff)
    d = d[~np.isnan(d)]
    obs = d.mean()
    n = len(d)
    signs = rng.choice([-1, 1], size=(n_perm, n))
    null = (signs * d).mean(axis=1)
    if alternative == 'less':
        p = (np.sum(null <= obs) + 1) / (n_perm + 1)
    elif alternative == 'greater':
        p = (np.sum(null >= obs) + 1) / (n_perm + 1)
    else:
        p = (np.sum(np.abs(null) >= abs(obs)) + 1) / (n_perm + 1)
    return obs, p


def imputation_footprint(y2_df, dv, predictor_cols):
    """For each outcome, count Y2 subjects (those with non-missing y) who have
    at least one missing predictor. Identical to the primary notebook."""
    y2_use = y2_df.dropna(subset=[dv])
    sub = y2_use[list(predictor_cols)]
    n_total = len(sub)
    any_missing = int(sub.isna().any(axis=1).sum())
    return {
        'dv': dv,
        'n_y2_with_y': int(n_total),
        'n_y2_any_imputed_predictor': any_missing,
        'pct_any_imputed_predictor': float(any_missing / n_total * 100) if n_total else 0.0,
        'per_column_missing': sub.isna().sum().to_dict(),
    }

In [6]:
def run_y1y2_compare(y1_df, y2_df, dv, id_col,
                     numeric_full, categorical_full,
                     numeric_demo, categorical_demo,
                     numeric_driving, categorical_driving,
                     l1_ratios=None, n_alphas=100, save_path=None):
    """Fit four ElasticNet models on Y1, evaluate on Y2, and run paired tests
    on per-subject squared error. Parallel to the primary notebook's
    `run_y1y2_compare`, with chosen alpha *and* l1_ratio reported.

    Models:
      full     = driving + demographics
      driving  = driving only
      demo     = demographics only
      baseline = constant = Y1 training mean

    Substantive paired comparisons (one-tailed, smaller error in first model):
      driving vs baseline -- does driving predict at all?
      demo    vs baseline -- do demographics predict at all?
      full    vs demo     -- does driving add beyond demographics?
      full    vs driving  -- do demographics add beyond driving?
    """
    full_res    = train_y1_predict_y2(y1_df, y2_df, dv, id_col,
                                      numeric_full,    categorical_full,
                                      l1_ratios=l1_ratios, n_alphas=n_alphas)
    driving_res = train_y1_predict_y2(y1_df, y2_df, dv, id_col,
                                      numeric_driving, categorical_driving,
                                      l1_ratios=l1_ratios, n_alphas=n_alphas)
    demo_res    = train_y1_predict_y2(y1_df, y2_df, dv, id_col,
                                      numeric_demo, categorical_demo,
                                      l1_ratios=l1_ratios, n_alphas=n_alphas)
    baseline_res = baseline_y1_predict_y2(y1_df, y2_df, dv, id_col)

    merged = (full_res.rename(columns={'sq_err': 'sq_err_full'})[['subject', 'sq_err_full']]
              .merge(driving_res.rename(columns={'sq_err': 'sq_err_driving'})[['subject', 'sq_err_driving']],   on='subject')
              .merge(demo_res.rename(columns={'sq_err': 'sq_err_demo'})[['subject', 'sq_err_demo']],            on='subject')
              .merge(baseline_res.rename(columns={'sq_err': 'sq_err_baseline'})[['subject', 'sq_err_baseline']],on='subject'))

    comparisons = [
        ('driving',  'baseline'),
        ('demo',     'baseline'),
        ('full',     'demo'),
        ('full',     'driving'),
    ]

    summary = {
        'dv': dv,
        'n_y2': int(merged.shape[0]),
        'mse_full':     float(merged['sq_err_full'].mean()),
        'mse_driving':  float(merged['sq_err_driving'].mean()),
        'mse_demo':     float(merged['sq_err_demo'].mean()),
        'mse_baseline': float(merged['sq_err_baseline'].mean()),
        'alpha_full':       float(full_res['alpha'].iloc[0]),
        'alpha_driving':    float(driving_res['alpha'].iloc[0]),
        'alpha_demo':       float(demo_res['alpha'].iloc[0]),
        'l1_ratio_full':    float(full_res['l1_ratio'].iloc[0]),
        'l1_ratio_driving': float(driving_res['l1_ratio'].iloc[0]),
        'l1_ratio_demo':    float(demo_res['l1_ratio'].iloc[0]),
    }

    print(f"\n=== Y1 -> Y2 generalization (ElasticNet): DV = {dv} (N_Y2 = {summary['n_y2']}) ===")
    print(f"  MSE_full     = {summary['mse_full']:.4f}    (alpha = {summary['alpha_full']:.4g}, l1 = {summary['l1_ratio_full']:.2f})")
    print(f"  MSE_driving  = {summary['mse_driving']:.4f}    (alpha = {summary['alpha_driving']:.4g}, l1 = {summary['l1_ratio_driving']:.2f})")
    print(f"  MSE_demo     = {summary['mse_demo']:.4f}    (alpha = {summary['alpha_demo']:.4g}, l1 = {summary['l1_ratio_demo']:.2f})")
    print(f"  MSE_baseline = {summary['mse_baseline']:.4f}")

    for a, b in comparisons:
        diff = merged[f'sq_err_{a}'] - merged[f'sq_err_{b}']
        diff_valid = diff.dropna()
        # Handle the zero-variance case (e.g., ElasticNet zeroed all driving
        # coefficients, making predictions = training mean = baseline). The
        # t-test is undefined; the permutation test correctly returns p = 1.
        if diff_valid.var(ddof=1) == 0 or len(diff_valid) < 2:
            t_stat, p_t = np.nan, np.nan
            obs = float(diff_valid.mean()) if len(diff_valid) else np.nan
            _, p_perm = paired_perm_test_mean_diff(diff_valid.values, alternative='less')
        else:
            t_stat, p_t = stats.ttest_1samp(diff_valid, 0, alternative='less')
            obs, p_perm = paired_perm_test_mean_diff(diff_valid.values, alternative='less')
        summary[f't_{a}_vs_{b}']         = float(t_stat) if t_stat == t_stat else np.nan
        summary[f'p_{a}_vs_{b}']         = float(p_t)    if p_t    == p_t    else np.nan
        summary[f'perm_p_{a}_vs_{b}']    = float(p_perm)
        summary[f'mean_diff_{a}_vs_{b}'] = float(obs)    if obs    == obs    else np.nan
        tag_t = f"{p_t:.4g}"   if p_t    == p_t    else "NaN (zero-variance)"
        tag_tstat = f"{t_stat:.3f}" if t_stat == t_stat else "NaN"
        print(f"  {a:<7s} vs {b:<8s}: mean diff = {obs:+.4f}, t = {tag_tstat}, t-p = {tag_t}, perm-p = {p_perm:.4g}")

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        merged.to_csv(f'{save_path}/{dv}_per_subject.csv', index=False)

    return merged, summary

In [7]:
# ElasticNetCV uses a data-driven alpha path per l1_ratio; passing the integer
# `n_alphas=100` requests 100 auto-generated alphas. The chosen alpha and
# l1_ratio per model are reported in the summary -- watch for chosen l1_ratio
# pinned at 0.99 across many outcomes (a hint that the L1 component is doing
# most of the work and the grid could be extended toward pure lasso).
l1_ratios_grid = [0.10, 0.30, 0.50, 0.70, 0.90, 0.95, 0.99]

# Predictor sets for the four models.
full_num, full_cat       = ivs + demos_num, demos_cat
driving_num, driving_cat = ivs, []
demos_num_only, demos_cat_only = demos_num, demos_cat
# baseline uses no predictors

# save per-subject squared errors
PER_SUBJECT_DIR = 'output/enet_errors/'
os.makedirs(PER_SUBJECT_DIR, exist_ok=True)

y1y2_per_subject = {}
y1y2_summary     = []
imputation_rows  = []

for dv in dvs:
    per_sub, summary = run_y1y2_compare(
        y1, y2, dv, id_col='XID',
        numeric_full=full_num,         categorical_full=full_cat,
        numeric_demo=demos_num_only,   categorical_demo=demos_cat_only,
        numeric_driving=driving_num,   categorical_driving=driving_cat,
        l1_ratios=l1_ratios_grid, n_alphas=100,
        save_path=PER_SUBJECT_DIR)
    y1y2_per_subject[dv] = per_sub
    y1y2_summary.append(summary)

    # Imputation footprint for the full-model predictor set (the broadest)
    imp = imputation_footprint(y2, dv, full_num + full_cat)
    imputation_rows.append(imp)

summary_df = pd.DataFrame(y1y2_summary)

# FDR-correct (Benjamini-Yekutieli) within each comparison family across the
# 12 DVs -- identical method to the primary notebook. NaN p-values (from
# zero-variance comparisons) propagate through multipletests; we re-run only
# on the non-NaN entries and re-insert NaN positions.
comparison_pairs = [('driving','baseline'),
                    ('demo','baseline'),
                    ('full','demo'),
                    ('full','driving')]
for a, b in comparison_pairs:
    for prefix in ('p', 'perm_p'):
        col = f'{prefix}_{a}_vs_{b}'
        vals = summary_df[col].values
        adj  = np.full(len(vals), np.nan)
        mask = ~np.isnan(vals)
        if mask.sum() > 0:
            adj[mask] = multipletests(vals[mask], method='fdr_by')[1]
        summary_df[col + '_fdr'] = adj

summary_df.to_csv('output/enet_summary.csv', index=False)
summary_df


=== Y1 -> Y2 generalization (ElasticNet): DV = LIFESATISFACTION (N_Y2 = 2704) ===
  MSE_full     = 0.3451    (alpha = 0.01694, l1 = 0.10)
  MSE_driving  = 0.3805    (alpha = 0.01398, l1 = 0.10)
  MSE_demo     = 0.3549    (alpha = 0.02088, l1 = 0.10)
  MSE_baseline = 0.4000
  driving vs baseline: mean diff = -0.0194, t = -5.504, t-p = 2.032e-08, perm-p = 9.999e-05
  demo    vs baseline: mean diff = -0.0450, t = -9.562, t-p = 1.254e-21, perm-p = 9.999e-05
  full    vs demo    : mean diff = -0.0098, t = -3.836, t-p = 6.395e-05, perm-p = 0.0002
  full    vs driving : mean diff = -0.0354, t = -8.528, t-p = 1.223e-17, perm-p = 9.999e-05

=== Y1 -> Y2 generalization (ElasticNet): DV = cog (N_Y2 = 2702) ===
  MSE_full     = 0.3904    (alpha = 0.0007888, l1 = 0.70)
  MSE_driving  = 0.4073    (alpha = 0.002589, l1 = 0.99)
  MSE_demo     = 0.3895    (alpha = 0.0007515, l1 = 0.70)
  MSE_baseline = 0.4282
  driving vs baseline: mean diff = -0.0209, t = -6.126, t-p = 5.17e-10, perm-p = 9.999e-05
  

,dv,n_y2,mse_full,mse_driving,mse_demo,mse_baseline,alpha_full,alpha_driving,alpha_demo,l1_ratio_full,...,perm_p_full_vs_driving,mean_diff_full_vs_driving,p_driving_vs_baseline_fdr,perm_p_driving_vs_baseline_fdr,p_demo_vs_baseline_fdr,perm_p_demo_vs_baseline_fdr,p_full_vs_demo_fdr,perm_p_full_vs_demo_fdr,p_full_vs_driving_fdr,perm_p_full_vs_driving_fdr
0,LIFESATISFACTION,2704,0.345133,0.380535,0.354925,0.399960,0.016936,0.013984,0.020879,0.10,...,0.0001,-0.035402,3.375261e-07,0.001241,4.669774e-20,0.000465,0.001191,0.003723,4.552582e-16,0.000621
1,cog,2702,0.390354,0.407274,0.389477,0.428151,0.000789,0.002589,0.000751,0.70,...,0.0002,-0.016920,1.717435e-08,0.001241,4.059485e-19,0.000465,1.000000,1.000000,8.320761e-07,0.000827
2,phys,2690,0.393281,0.414097,0.413045,0.434441,0.001579,0.023475,0.003742,0.99,...,0.0001,-0.020815,5.469743e-06,0.001241,6.121026e-08,0.000465,0.000016,0.003723,1.010401e-08,0.000621
3,fati,2700,0.571447,0.584970,0.581427,0.592251,0.010289,0.033551,0.007862,0.10,...,0.0001,-0.013523,7.934689e-03,0.011915,7.068990e-06,0.000465,0.002134,0.003723,4.166413e-07,0.000621
4,socialrole,2659,0.426774,0.430488,0.431128,0.436542,0.003592,0.016506,0.006468,0.99,...,0.0018,-0.003714,8.328320e-02,0.101155,3.776400e-04,0.000745,0.395855,0.407721,5.770606e-03,0.006093
5,iso,2687,0.400651,0.406635,0.401872,0.409938,0.008404,0.058192,0.003443,0.50,...,0.0030,-0.005984,1.640043e-01,0.187570,1.306020e-03,0.001862,1.000000,1.000000,8.044272e-03,0.009309
6,info,2695,0.628860,0.664893,0.628816,0.669345,0.001998,0.001593,0.001413,0.70,...,0.0001,-0.036033,3.774637e-01,0.422446,3.876262e-10,0.000465,1.000000,1.000000,5.258446e-10,0.000621
7,emo,2698,0.520208,0.536310,0.520183,0.540764,0.007560,0.026002,0.007427,0.30,...,0.0001,-0.016102,2.615854e-01,0.284226,5.459604e-06,0.000465,1.000000,1.000000,1.055664e-05,0.000621
8,ins,2647,0.577486,0.646326,0.577215,0.660061,0.002384,0.003283,0.001030,0.99,...,0.0001,-0.068840,5.140861e-03,0.003723,5.547017e-14,0.000465,1.000000,1.000000,1.384852e-12,0.000621
9,dep,2702,0.184787,0.187185,0.185675,0.188403,0.006129,0.007138,0.006762,0.95,...,0.0002,-0.002399,1.133347e-01,0.134577,1.279769e-04,0.000745,0.425493,0.414051,2.773850e-04,0.000827
